# Task 20 — Memory Management & Garbage Collection (Detailed)

---

# Table of Contents

1. Introduction
2. Python Memory Model
3. Stack vs Heap Memory
4. Reference Counting
5. Garbage Collection
6. The `gc` Module
7. Weak References
8. Weak Containers
9. Common Memory Pitfalls
10. Best Practices
11. Interview Questions
12. Practice Problems
13. Chapter Cheat Sheet

---

# 1. Introduction

Memory management is the process of allocating, using, and freeing memory while a Python program is running.

Unlike languages such as C or C++, Python automatically manages memory, allowing developers to focus on writing logic instead of manually allocating and freeing memory.

Python uses **two primary mechanisms**:

- Reference Counting
- Garbage Collection (GC)

Understanding these concepts helps explain:

- Why objects disappear
- Why some objects remain in memory
- How circular references are handled
- Why memory leaks can still occur

---

# 2. Python Memory Model

One of the most important ideas in Python is:

> Variables **do not store objects**. They store **references** to objects.

Example:

```python
a = [1, 2, 3]
```

Memory:

```
a ─────────► [1,2,3]
```

Now:

```python
b = a
```

```
a ─┐
   │
   ▼
 [1,2,3]
   ▲
   │
b ─┘
```

No new list is created.

Only another reference is added.

---

# 3. Stack vs Heap Memory

## Heap

The Heap stores **all Python objects**.

Examples:

- int
- float
- str
- list
- tuple
- dict
- functions
- classes
- instances

Example

```python
nums = [1, 2, 3]
```

```
Heap

List Object
┌──────────┐
│1│2│3│
└──────────┘
```

---

## Stack

Python creates a **stack frame** whenever a function is called.

Example

```python
def add(a, b):
    c = a + b
    return c

add(5, 10)
```

During execution

```
Call Stack

┌─────────────────┐
│ add() Frame     │
│ a → 5           │
│ b →10           │
│ c →15           │
└─────────────────┘
```

The frame stores **references**, not the actual objects.

After the function returns, the frame is destroyed.

---

## Stack vs Heap

| Stack | Heap |
|--------|------|
| Stores execution frames | Stores Python objects |
| Fast allocation | Dynamic allocation |
| Temporary | Objects live until unreferenced |
| Follows LIFO | No fixed order |

---

# 4. Reference Counting

Every Python object has a **reference count**.

Python increases it whenever another reference points to the object.

Example

```python
a = [1, 2]

b = a
c = a
```

Memory

```
a ─┐
b ─┼────► [1,2]
c ─┘

Reference Count = 3
```

---

Removing references

```python
del b
```

Reference Count

```
2
```

Object still exists.

---

When the last reference disappears

```python
del a
del c
```

Reference Count becomes

```
0
```

CPython immediately destroys the object (unless it is part of a circular reference).

---

## Viewing Reference Count

```python
import sys

a = []

print(sys.getrefcount(a))
```

Note:

`getrefcount()` itself temporarily increases the count by **1**.

Never depend on the exact number.

Instead observe whether the count changes.

---

# 5. Garbage Collection

Reference counting cannot solve circular references.

Example

```python
a = []
b = []

a.append(b)
b.append(a)
```

Memory

```
a ───► b
▲      │
└──────┘
```

Even after

```python
del a
del b
```

the two list objects still reference each other.

Reference counting cannot free them.

Python's **Garbage Collector (GC)** periodically detects such unreachable cycles and removes them.

---

## Python Uses Both

| Mechanism | Purpose |
|-----------|---------|
| Reference Counting | Immediate cleanup |
| Garbage Collector | Cleans circular references |

---

# 6. The `gc` Module

Import

```python
import gc
```

---

## Collect Garbage

```python
gc.collect()
```

Runs the garbage collector manually.

Returns the number of unreachable objects collected.

---

## Disable GC

```python
gc.disable()
```

Reference counting still works.

Only cyclic garbage collection stops.

---

## Enable Again

```python
gc.enable()
```

---

## Check Status

```python
gc.isenabled()
```

Returns

```python
True
```

---

## Statistics

```python
gc.get_stats()
```

Returns collection statistics.

Mostly useful for debugging and profiling.

---

# 7. Weak References

Normally,

```python
a = obj
```

creates a **strong reference**.

Strong references keep objects alive.

A **weak reference** does **not**.

Example

```python
import weakref

class Person:
    pass

p = Person()

w = weakref.ref(p)
```

Memory

```
p ─────► Person()

w - - -> Person()
```

Notice the dotted arrow.

It doesn't increase the reference count.

---

Access

```python
print(w())
```

Output

```
<__main__.Person object at ...>
```

After

```python
del p
```

Now

```python
print(w())
```

Output

```
None
```

The object no longer exists.

---

# Strong vs Weak References

| Strong | Weak |
|---------|------|
| Increases reference count | Doesn't increase count |
| Keeps object alive | Doesn't keep object alive |
| Normal variable | `weakref.ref()` |

---

# 8. Weak Containers

Python provides useful weak-reference collections.

---

## WeakValueDictionary

Stores **weak values**.

Objects disappear automatically when no strong references remain.

---

## WeakKeyDictionary

Stores **weak keys**.

Entries disappear when keys are destroyed.

---

## WeakSet

A set storing weak references.

Commonly used in frameworks and observer patterns.

---

# 9. Common Memory Pitfalls

## 1. Mistaking Assignment for Copy

```python
a = [1,2]
b = a
```

Both refer to the same list.

---

## 2. Large Global Objects

Large global variables remain alive until the program ends.

---

## 3. Circular References

Objects referencing each other may require the garbage collector.

---

## 4. Holding Unnecessary References

Keeping references in caches, lists, or dictionaries prevents objects from being destroyed.

---

# 10. Best Practices

✔ Prefer local variables over unnecessary globals.

✔ Remove references when objects are no longer needed.

✔ Don't manually call `gc.collect()` unless debugging or profiling.

✔ Use weak references for caches and observer systems.

✔ Understand that assignment copies references, not objects.

---

# 11. Interview Questions

### Q1. What is reference counting?

A mechanism that keeps track of how many references point to an object. When the count reaches zero, the object is destroyed.

---

### Q2. Why does Python also need a Garbage Collector?

Because reference counting cannot clean circular references.

---

### Q3. Does `gc.disable()` stop reference counting?

No.

It only disables cyclic garbage collection.

---

### Q4. What does `gc.collect()` return?

The number of unreachable objects collected.

---

### Q5. What is a weak reference?

A reference that does **not** increase an object's reference count.

---

### Q6. Where are Python objects stored?

On the Heap.

---

### Q7. What is stored inside a stack frame?

References to objects and function execution state.

---

# 12. Practice Problems

## Easy

1. Explain Stack vs Heap.
2. Explain Reference Counting.
3. Difference between Reference Counting and Garbage Collection.
4. What is `gc.collect()`?
5. What is a Weak Reference?

---

## Medium

6. Demonstrate circular references.

7. Use `sys.getrefcount()` to observe reference count changes.

8. Disable GC and verify with `gc.isenabled()`.

9. Create and use a weak reference.

10. Compare strong and weak references using an example.

---

# 13. Chapter Cheat Sheet

## Memory Flow

```
Variable
    │
    ▼
Reference
    │
    ▼
Heap Object
```

---

## Object Lifetime

```
Reference Count > 0
        │
        ▼
 Object Lives

Reference Count = 0
        │
        ▼
Destroyed (Reference Counting)

Circular Reference
        │
        ▼
Garbage Collector
```

---

## Important Modules

```python
import sys
import gc
import weakref
```

---

## Important Functions

```python
sys.getrefcount(obj)

gc.collect()

gc.enable()

gc.disable()

gc.isenabled()

weakref.ref(obj)
```

---

# Summary

After completing this chapter you should understand:

- How Python stores objects in memory
- Difference between Stack and Heap
- How Reference Counting works
- Why Garbage Collection is required
- How to use the `gc` module
- What Weak References are and when they are useful
- Common memory-related pitfalls and best practices

---

